In [1]:
!git clone https://github.com/clayton-h-costa/pv_fault_dataset.git

Cloning into 'pv_fault_dataset'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 17 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 26.69 MiB | 5.88 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [23]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from scipy.io import loadmat
warnings.filterwarnings('ignore')

# Set ggplot style
plt.style.use('ggplot')

# ML and Deep Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    cohen_kappa_score,
    log_loss,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)
from sklearn.ensemble import RandomForestClassifier
import shap
import joblib

In [4]:
# Load the dataset
elec_data = loadmat("pv_fault_dataset/dataset_elec.mat")
amb_data  = loadmat("pv_fault_dataset/dataset_amb.mat")

In [5]:
# Extract variables
vdc1 = elec_data['vdc1'].flatten()
vdc2 = elec_data['vdc2'].flatten()
idc1 = elec_data['idc1'].flatten()
idc2 = elec_data['idc2'].flatten()

irr = amb_data['irr'].flatten()
pvt = amb_data['pvt'].flatten()
f_nv = amb_data['f_nv'].flatten()

# Create a DataFrame for easier analysis
df = pd.DataFrame({
    'vdc1': vdc1,
    'vdc2': vdc2,
    'idc1': idc1,
    'idc2': idc2,
    'irradiance': irr,
    'temperature': pvt,
    'fault_label': f_nv
})

df = df[df['fault_label'] != 2].copy()

In [6]:
# View the features
df.head()

,vdc1,vdc2,idc1,idc2,irradiance,temperature,fault_label
0,0.7143,0.5550,0.0608,0.0073,1.3729,2.3816,0
1,0.6944,0.5427,0.0615,0.0064,1.3604,2.3816,0
2,0.7110,0.5583,0.0646,0.0067,1.5118,2.3883,0
3,0.6991,0.5465,0.0628,0.0067,1.5534,2.3920,0
4,0.7035,0.5553,0.0606,0.0076,1.4355,2.3920,0


In [7]:
# Total samples
print("Total samples:", len(df))

Total samples: 1363427


In [8]:
# Label value counts
df['fault_label'].value_counts()

fault_label
0    1162931
4     188473
3       6024
1       5999
Name: count, dtype: int64

In [9]:
# **3. Data Preprocessing**

# Add calculated columns
df['power_string1'] = df['vdc1'] * df['idc1']
df['power_string2'] = df['vdc2'] * df['idc2']
df['total_power'] = df['power_string1'] + df['power_string2']
df['voltage_ratio'] = df['vdc1'] / df['vdc2']
df['current_ratio'] = df['idc1'] / df['idc2']

print(f"Dataset shape: {df.shape}")
print(f"Number of samples: {len(df)}")

# Create fault name mapping
fault_names = {
    0: 'Normal Operation',
    1: 'Short-Circuit',
    3: 'Open Circuit',
    4: 'Shadowing'
}

# Add fault name column
df['fault_label'] = df['fault_label'].map(fault_names)

Dataset shape: (1363427, 12)
Number of samples: 1363427


In [10]:
df.head()

,vdc1,vdc2,idc1,idc2,irradiance,temperature,fault_label,power_string1,power_string2,total_power,voltage_ratio,current_ratio
0,0.7143,0.5550,0.0608,0.0073,1.3729,2.3816,Normal Operation,0.043429,0.004052,0.047481,1.287027,8.328767
1,0.6944,0.5427,0.0615,0.0064,1.3604,2.3816,Normal Operation,0.042706,0.003473,0.046179,1.279528,9.609375
2,0.7110,0.5583,0.0646,0.0067,1.5118,2.3883,Normal Operation,0.045931,0.003741,0.049671,1.273509,9.641791
3,0.6991,0.5465,0.0628,0.0067,1.5534,2.3920,Normal Operation,0.043903,0.003662,0.047565,1.279231,9.373134
4,0.7035,0.5553,0.0606,0.0076,1.4355,2.3920,Normal Operation,0.042632,0.004220,0.046852,1.266883,7.973684


In [ ]:
# **4. Train Test Splitting**

# Seperate data
X = df.drop(['fault_label'], axis=1).values
y = df['fault_label'].values

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (1363427, 11)
y Shape: (1363427,)


In [12]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [13]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
# Show the shapes
print("X train shape:", X_train_scaled.shape)
print("X test shape:", X_test_scaled.shape)

X train shape: (1090741, 11)
X test shape: (272686, 11)


In [15]:
# View total samples
print("Training set:", len(X_train_scaled),"samples")
print("Testing set:", len(X_test_scaled),"samples")

Training set: 1090741 samples
Testing set: 272686 samples


In [16]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))
print("Class weights:", class_weights)

Class weights: {'Normal Operation': np.float64(0.2931015301866836), 'Open Circuit': np.float64(56.585443037974684), 'Shadowing': np.float64(1.808509474131013), 'Short-Circuit': np.float64(56.82126484684309)}


In [17]:
# **6. Training the Neural Network**

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight=class_weights
)

rf.fit(X_train_scaled, y_train)

RandomForestClassifier(class_weight={'Normal Operation': np.float64(0.2931015301866836),
                                     'Open Circuit': np.float64(56.585443037974684),
                                     'Shadowing': np.float64(1.808509474131013),
                                     'Short-Circuit': np.float64(56.82126484684309)},
                       n_jobs=-1, random_state=42)

In [18]:
# Get predictions
y_pred = rf.predict(X_test_scaled)
y_proba = rf.predict_proba(X_test_scaled)

In [19]:
class_order = rf.classes_
print("Class order:", class_order)

Class order: ['Normal Operation' 'Open Circuit' 'Shadowing' 'Short-Circuit']


In [20]:
acc = accuracy_score(y_test, y_pred)
bacc = balanced_accuracy_score(y_test, y_pred)

In [21]:
prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0
)

In [24]:
kappa = cohen_kappa_score(y_test, y_pred)

# Log Loss needs proba + true labels
ll = log_loss(y_test, y_proba, labels=class_order)

# For ROC-AUC + PR-AUC (macro), we need binarized y
y_test_bin = label_binarize(y_test, classes=class_order)

In [25]:
roc_auc_macro = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
pr_auc_macro  = average_precision_score(y_test_bin, y_proba, average="macro")

In [26]:
metrics_dict = {
    "accuracy": float(acc),
    "balanced_accuracy": float(bacc),
    "precision_macro": float(prec_macro),
    "recall_macro": float(rec_macro),
    "f1_macro": float(f1_macro),
    "cohens_kappa": float(kappa),
    "log_loss": float(ll),
    "roc_auc_macro_ovr": float(roc_auc_macro),
    "pr_auc_macro": float(pr_auc_macro)
}

In [27]:
print("\nRandom Forest: Evaluation Metrics:")
for k, v in metrics_dict.items():
    print(f"{k:20s}: {v:.6f}")


Random Forest: Evaluation Metrics:
accuracy            : 0.999186
balanced_accuracy   : 0.998036
precision_macro     : 0.999008
recall_macro        : 0.998036
f1_macro            : 0.998521
cohens_kappa        : 0.996785
log_loss            : 0.002479
roc_auc_macro_ovr   : 0.999996
pr_auc_macro        : 0.999979


In [ ]:
# Visualize confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

In [ ]:
# View classification report
print(classification_report(y_true, y_pred))

In [ ]:
# Best model predictions
y_pred_proba_best = best_model.predict(X_test, verbose=0)
y_pred_best = np.argmax(y_pred_proba_best, axis=1)

In [ ]:
# Show the history
display_history(history)

In [ ]:
# Display confusion matrix
cm = confusion_matrix(y_true, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

In [ ]:
# Classification report
print(classification_report(y_true, y_pred_best))

In [ ]:
# Evaluate on test set
test_results = best_model.evaluate(X_test, y_test, verbose=0)

In [ ]:
metrics = ['Loss', 'Accuracy', 'Precision', 'Recall', 'AUC']

df_results = pd.DataFrame(
  [np.round(test_results, 3)],
  columns=[f'Test {m}' for m in metrics]
)
df_results

In [ ]:


# **11. SHAP Interpretability**

# Background for shap
background = shap.sample(X_train, 100)

# Create the explainer
explainer = shap.KernelExplainer(best_model.predict, background)

# Generate SHAP values
shap_vals = explainer.shap_values(X_test[:100]) # for speed efficiency, keep it to 100.

# Feature names
feats = df.drop(['fault_label'], axis=1).columns

# Summary plot
shap.summary_plot(
    shap_vals,
    X_test[:100],
    feature_names=feats,
    plot_type='bar'
)
plt.tight_layout()
plt.show()

# Force plot for a single prediction
shap.initjs()
shap.force_plot(explainer.expected_value[0], # Base value for the first output class
                shap_vals[0, :, 0],   # Take the SHAP values for the first sample, all features, first output class
                X_test[0],            # Actual feature values for the first sample
                feature_names=feats)

# **12. Save Best Model**

joblib.dump(scaler, 'scaler.pkl')

with open("history.pkl", 'wb') as f:
    joblib.dump(history, f)

best_model.save('best_neural_network.keras')